# 03: HyDE + Few-Shot + Type Boost (BM25-based)

**Upgrade from notebook 02** — adds 3 toggleable features on top of the same BM25 agent:

| Feature | Toggle | What it does |
|---------|--------|-------------|
| **A: HyDE** |  | LLM generates hypothetical German article → uses it as BM25 query |
| **B: Few-Shot Bank** |  | Domain-matched examples from train.csv guide HyDE generation |
| **C: Type Boost** |  | Soft 1.5× score boost for documents matching detected legal type |

**What stays from notebook 02:** BM25 keyword search, ReAct agent, Mistral-7B LLM, same corpus

**What does NOT go here (saved for 03_hyde_kaggle):**
- ❌ FAISS embeddings / semantic search
- ❌ RRF fusion (BM25 + FAISS)
- ❌ GBNF grammar-constrained agent output
- ❌ Reranker (cross-encoder or Qwen3)

**Ablation testing:** Toggle features in CONFIG to measure individual impact.


In [1]:
# === INSTALL DEPENDENCIES ===
!pip install -q rank-bm25 pandas tqdm
!pip install -q llama-cpp-python --prefer-binary \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 GB 478.1 kB/s eta 0:00:000:01m0:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.4 MB/s eta 0:00:00


## 1. Setup & Configuration

In [2]:
import os
import sys
from pathlib import Path

# === CONFIGURATION ===
# Choose which dataset to run on: "val" or "test"
DATASET_MODE = "test"  # Change to "test" for final submission

# Set to True to rebuild indices from CSV (required on first run)
# Set to False to load cached indices (faster for subsequent runs)
FORCE_REBUILD_INDICES = False

# Detect environment
KAGGLE_ENV = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if KAGGLE_ENV:
    # Kaggle paths
    DATA_PATH = Path("/kaggle/input/competitions/llm-agentic-legal-information-retrieval")
    MODEL_PATH = Path("/kaggle/input/datasets/charan1996/mistral-7b-gguf")
    OUTPUT_PATH = Path("/kaggle/working")
    INDEX_PATH = Path("/kaggle/working/cache")
    # (no external utils needed)
else:
    # Local development paths
    REPO_ROOT = Path(".").resolve().parent
    DATA_PATH = REPO_ROOT / "data"
    MODEL_PATH = REPO_ROOT / "models"
    OUTPUT_PATH = REPO_ROOT / "output"
    INDEX_PATH = REPO_ROOT / "data" / "processed"

# CSV corpus files for index building
LAWS_CSV = DATA_PATH / "laws_de.csv"
COURTS_CSV = DATA_PATH / "court_considerations.csv"

# Index cache paths
LAWS_INDEX_PATH = INDEX_PATH / "laws_index.pkl"
COURTS_INDEX_PATH = INDEX_PATH / "courts_index.pkl"

# Derived paths based on DATASET_MODE
QUERY_FILE = DATA_PATH / f"{DATASET_MODE}.csv"
IS_VALIDATION_MODE = DATASET_MODE == "val"

# Create output directory
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
INDEX_PATH.mkdir(parents=True, exist_ok=True)

print(f"Environment: {'Kaggle' if KAGGLE_ENV else 'Local'}")
print(f"Dataset mode: {DATASET_MODE}")
print(f"Query file: {QUERY_FILE}")
print(f"Validation mode: {IS_VALIDATION_MODE}")
print(f"Force rebuild indices: {FORCE_REBUILD_INDICES}")
print(f"\nCorpus files:")
print(f"  Laws CSV: {LAWS_CSV} ({LAWS_CSV.stat().st_size / 1e6:.1f} MB)" if LAWS_CSV.exists() else f"  Laws CSV: {LAWS_CSV} (NOT FOUND)")
print(f"  Courts CSV: {COURTS_CSV} ({COURTS_CSV.stat().st_size / 1e9:.2f} GB)" if COURTS_CSV.exists() else f"  Courts CSV: {COURTS_CSV} (NOT FOUND)")
print(f"\nIndex cache: {INDEX_PATH}")

Environment: Kaggle
Dataset mode: test
Query file: /kaggle/input/competitions/llm-agentic-legal-information-retrieval/test.csv
Validation mode: False
Force rebuild indices: False

Corpus files:
  Laws CSV: /kaggle/input/competitions/llm-agentic-legal-information-retrieval/laws_de.csv (73.0 MB)
  Courts CSV: /kaggle/input/competitions/llm-agentic-legal-information-retrieval/court_considerations.csv (2.43 GB)

Index cache: /kaggle/working/cache


In [3]:
# Configuration — with FEATURE TOGGLES for ablation testing
# Toggle each feature ON/OFF to measure its individual impact on F1
CONFIG = {
    # Model settings
    "model_file": "mistral-7b-instruct-v0.2.Q4_K_M.gguf",
    "n_ctx": 8192,
    "n_threads": 8,
    "n_gpu_layers": -1,    # -1 = all on GPU (was CPU before!)
    
    # Agent settings
    "max_iterations": 3,
    "max_tokens": 512,
    "temperature": 0.1,
    "max_observation_chars": 1200,
    "max_conversation_chars": 28000,
    
    # Retrieval settings
    "top_k_laws": 40,
    "top_k_courts": 40,
    
    # ══════════════════════════════════════════════════════════
    # FEATURE TOGGLES — flip these to test each feature in isolation
    # ══════════════════════════════════════════════════════════
    
    # Feature A: HyDE (hypothetical document generation)
    "hyde_enabled": False,           # False = pure BM25 (like notebook 02)
    "hyde_max_tokens": 300,
    "hyde_temperature": 0.3,
    "hyde_few_shot_count": 3,
    "hyde_target_chars_law": 300,
    "hyde_target_chars_court": 400,
    
    # Feature B: Few-Shot Bank (domain-matched examples)
    "few_shot_enabled": False,       # False = HyDE without examples (generic prompt)
    "hyde_examples_per_type": 3,
    "hyde_max_synthetic_types": 50,
    
    # Feature C: Type Boost (hierarchical search)
    "type_boost_enabled": False,     # False = no type-based score boosting
    "type_boost_factor": 1.5,
    
    # Feature D: Prompt Injection (type registry appended to agent prompt)
    "prompt_injection_enabled": False,
    
    # Feature E: CCH Output Formatting ([TYPE] prefix in tool observations)
    "cch_enabled": False,
    
    # ══════════════════════════════════════════════════════════
    # ABLATION TEST CONFIGURATIONS (uncomment one at a time):
    # ══════════════════════════════════════════════════════════
    # Test 1: Baseline (all features OFF) = same as notebook 02
    #   "hyde_enabled": False, "few_shot_enabled": False, "type_boost_enabled": False
    # Test 2: HyDE only (no few-shot, no type boost)
    #   "hyde_enabled": True, "few_shot_enabled": False, "type_boost_enabled": False
    # Test 3: HyDE + Few-Shot (no type boost)
    #   "hyde_enabled": True, "few_shot_enabled": True, "type_boost_enabled": False
    # Test 4: All features ON (current default)
    #   "hyde_enabled": True, "few_shot_enabled": True, "type_boost_enabled": True
    # Test 5: Type boost only (no HyDE, no few-shot)
    #   "hyde_enabled": False, "few_shot_enabled": False, "type_boost_enabled": True
    # Test 6: Prompt injection only (all retrieval features off)
    #   "hyde_enabled": False, "few_shot_enabled": False, "type_boost_enabled": False, "prompt_injection_enabled": True
    # Test 7: CCH output only (all retrieval features off)
    #   "hyde_enabled": False, "few_shot_enabled": False, "type_boost_enabled": False, "prompt_injection_enabled": False, "cch_enabled": True
}

print(f"Feature toggles:")
print(f"  HyDE:      {CONFIG['hyde_enabled']}")
print(f"  Few-Shot:  {CONFIG['few_shot_enabled']}")
print(f"  Type Boost:{CONFIG['type_boost_enabled']}")
print(f"  Prompt Inj:{CONFIG['prompt_injection_enabled']}")
print(f"  CCH:       {CONFIG['cch_enabled']}")


Feature toggles:
  HyDE:      False
  Few-Shot:  False
  Type Boost:False
  Prompt Inj:False
  CCH:       False


## 2. Load Corpora and Build/Load Indices

In [4]:
import pandas as pd
from tqdm.notebook import tqdm
import pickle
import re
from rank_bm25 import BM25Okapi


class BM25Index:
    """BM25 index for keyword search over legal documents.

    Supports Swiss federal laws (SR) and court decisions (BGE).
    """

    def __init__(
        self,
        documents: list[dict] | None = None,
        text_field: str = "text",
        citation_field: str = "citation",
    ):
        """Initialize BM25 index.

        Args:  
            documents: List of document dictionaries
            text_field: Key for document text in dict
            citation_field: Key for citation string in dict
        """
        self.text_field = text_field
        self.citation_field = citation_field

        self.documents: list[dict] = []
        self.index: BM25Okapi | None = None
        self._tokenized_corpus: list[list[str]] = []

        if documents:
            self.build(documents)

    def tokenize(self, text: str) -> list[str]:
        """Tokenize text for BM25 indexing.

        Simple whitespace + lowercase tokenization.
        Can be overridden for language-specific tokenization.

        Args:
            text: Text to tokenize

        Returns:
            List of tokens
        """
        # Lowercase and split on non-alphanumeric characters
        text = text.lower()
        tokens = re.split(r"\W+", text)
        # Filter empty tokens
        return [t for t in tokens if t]

    def build(self, documents: list[dict]) -> None:
        """Build BM25 index from documents.

        Args:
            documents: List of document dictionaries
        """
        self.documents = documents

        # Tokenize all documents
        self._tokenized_corpus = []
        for doc in documents:
            text = doc.get(self.text_field, "")
            tokens = self.tokenize(text)
            self._tokenized_corpus.append(tokens)

        # Build BM25 index
        self.index = BM25Okapi(self._tokenized_corpus)

    def search(
        self,
        query: str,
        top_k: int = 10,
        return_scores: bool = False,
    ) -> list[dict]:
        """Search the index with a query.

        Args:
            query: Search query string
            top_k: Number of results to return
            return_scores: Whether to include BM25 scores in results

        Returns:
            List of matching documents (with optional scores)
        """
        if self.index is None:
            raise ValueError("Index not built. Call build() first.")

        # Tokenize query
        query_tokens = self.tokenize(query)

        if not query_tokens:
            return []

        # Get BM25 scores
        scores = self.index.get_scores(query_tokens)

        # Get top-k indices
        top_indices = scores.argsort()[-top_k:][::-1]

        # Build results
        results = []
        for idx in top_indices:
            if scores[idx] <= 0:
                continue

            doc = self.documents[idx].copy()
            if return_scores:
                doc["_score"] = float(scores[idx])
            results.append(doc)

        return results

    def save(self, path: Path | str) -> None:
        """Save index to disk.

        Args:
            path: Path to save index (creates .pkl file)
        """
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)

        data = {
            "documents": self.documents,
            "tokenized_corpus": self._tokenized_corpus,
            "text_field": self.text_field,
            "citation_field": self.citation_field,
        }

        with open(path, "wb") as f:
            pickle.dump(data, f)

    @classmethod
    def load(cls, path: Path | str) -> "BM25Index":
        """Load index from disk.

        Args:
            path: Path to saved index

        Returns:
            Loaded BM25Index instance
        """
        path = Path(path)

        with open(path, "rb") as f:
            data = pickle.load(f)

        instance = cls(
            text_field=data["text_field"],
            citation_field=data.get("citation_field", "citation"),
        )
        instance.documents = data["documents"]
        instance._tokenized_corpus = data["tokenized_corpus"]
        instance.index = BM25Okapi(instance._tokenized_corpus)

        return instance


def load_csv_corpus(
    csv_path: Path,
    chunk_size: int = 100_000,
    max_rows: int | None = None
) -> list[dict]:
    """Load CSV corpus into list of dicts with progress bar.
    
    Args:
        csv_path: Path to CSV file with 'citation' and 'text' columns
        chunk_size: Rows to process per chunk (for memory efficiency)
        max_rows: Optional limit on rows (for testing with smaller corpus)
    
    Returns:
        List of {"citation": str, "text": str} dicts
    """
    documents = []
    
    # Count rows for progress bar (fast line count)
    print(f"Counting rows in {csv_path.name}...")
    with open(csv_path, encoding='utf-8') as f:
        total_rows = sum(1 for _ in f) - 1  # minus header
    
    if max_rows:
        total_rows = min(total_rows, max_rows)
    print(f"Total rows to load: {total_rows:,}")
    
    rows_loaded = 0
    with tqdm(total=total_rows, desc=f"Loading {csv_path.name}") as pbar:
        for chunk in pd.read_csv(csv_path, chunksize=chunk_size):
            for _, row in chunk.iterrows():
                if max_rows and rows_loaded >= max_rows:
                    break
                documents.append({
                    "citation": str(row["citation"]),
                    "text": str(row["text"]) if pd.notna(row["text"]) else ""
                })
                rows_loaded += 1
            pbar.update(min(len(chunk), total_rows - pbar.n))
            if max_rows and rows_loaded >= max_rows:
                break
    
    return documents


def get_or_build_index(
    name: str,
    csv_path: Path,
    index_path: Path,
    force_rebuild: bool = False,
    max_rows: int | None = None
) -> BM25Index:
    """Load cached index or build from CSV.
    
    Args:
        name: Index name for logging
        csv_path: Path to corpus CSV
        index_path: Path to cache index pickle
        force_rebuild: If True, rebuild even if cache exists
        max_rows: Optional row limit (for testing with smaller corpus)
    
    Returns:
        BM25Index instance
    """
    # Use cached index if available and not forcing rebuild
    if index_path.exists() and not force_rebuild:
        print(f"Loading cached {name} index from {index_path}")
        index = BM25Index.load(index_path)
        print(f"  Loaded {len(index.documents):,} documents")
        return index
    
    # Check CSV exists
    if not csv_path.exists():
        print(f"Warning: {csv_path} not found. Creating empty index.")
        return BM25Index(documents=[])
    
    # Load corpus from CSV
    print(f"\n{'='*50}")
    print(f"Building {name} index from {csv_path}")
    print(f"{'='*50}")
    documents = load_csv_corpus(csv_path, max_rows=max_rows)
    
    if not documents:
        print(f"Warning: No documents loaded. Creating empty index.")
        return BM25Index(documents=[])
    
    # Build BM25 index
    print(f"\nBuilding BM25 index for {len(documents):,} documents...")
    index = BM25Index(
        documents=documents,
        text_field="text",
        citation_field="citation"
    )
    print(f"Index built successfully!")
    
    # Cache index for future runs
    if not KAGGLE_ENV:
        print(f"Saving index to {index_path}...")
        index.save(index_path)
        print(f"Index cached.")
    
    return index

In [5]:
# Load or build laws index
# Laws CSV: ~45MB, ~269K rows
# Build time: ~30 seconds | Load from cache: <1 second

laws_index = get_or_build_index(
    name="laws",
    csv_path=LAWS_CSV,
    index_path=LAWS_INDEX_PATH,
    force_rebuild=FORCE_REBUILD_INDICES,
    # max_rows=10000  # Uncomment to test with smaller corpus
)
print(f"\nLaws index: {len(laws_index.documents):,} documents")

# Test search
test_results = laws_index.search("Vertrag", top_k=3)
print(f"\nTest search 'Vertrag': {len(test_results)} results")
if test_results:
    print(f"  Top result: {test_results[0].get('citation', 'N/A')}")


Building laws index from /kaggle/input/competitions/llm-agentic-legal-information-retrieval/laws_de.csv
Counting rows in laws_de.csv...
Total rows to load: 272,433


Loading laws_de.csv:   0%|          | 0/272433 [00:00<?, ?it/s]


Building BM25 index for 175,933 documents...
Index built successfully!

Laws index: 175,933 documents

Test search 'Vertrag': 3 results
  Top result: Art. 17 Abs. 6 VID


In [6]:
# Load or build courts index
# Courts CSV: ~2.3GB, ~2.5M rows
# Full corpus build time: ~15-20 minutes | Load from cache: ~10 seconds
# Full corpus can have peak memory during build: ~8-16GB

courts_index = get_or_build_index(
    name="courts",
    csv_path=COURTS_CSV,
    index_path=COURTS_INDEX_PATH,
    force_rebuild=FORCE_REBUILD_INDICES,
    max_rows=100000  # Change to use bigger corpus
)
print(f"\nCourts index: {len(courts_index.documents):,} documents")

# Test search
test_results = courts_index.search("Meinungsfreiheit", top_k=3)
print(f"\nTest search 'Meinungsfreiheit': {len(test_results)} results")
if test_results:
    print(f"  Top result: {test_results[0].get('citation', 'N/A')}")


Building courts index from /kaggle/input/competitions/llm-agentic-legal-information-retrieval/court_considerations.csv
Counting rows in court_considerations.csv...
Total rows to load: 100,000


Loading court_considerations.csv:   0%|          | 0/100000 [00:00<?, ?it/s]


Building BM25 index for 100,000 documents...
Index built successfully!

Courts index: 100,000 documents

Test search 'Meinungsfreiheit': 3 results
  Top result: BGE 148 I 33 E. 6.1


## 3. Define Search Tools

In [7]:
class LawSearchTool:
    """Tool for searching Swiss federal laws corpus.

    Searches the SR (Systematische Rechtssammlung) collection
    using BM25 keyword matching.
    """

    name: str = "search_laws"
    description: str = """Search Swiss federal laws (SR/Systematische Rechtssammlung) by keywords.
Input: Search query string (can be in German, French, Italian, or English)
Output: List of relevant law citations with text excerpts

Use this tool to find relevant federal law provisions for a legal question.
Example queries: "contract formation requirements", "Vertragsabschluss", "divorce grounds"
"""

    def __init__(
        self,
        index: BM25Index,
        top_k: int = 5,
        max_excerpt_length: int = 300,
    ):
        """Initialize law search tool.

        Args:
            index: BM25Index for federal laws corpus
            top_k: Number of results to return
            max_excerpt_length: Maximum characters for text excerpts
        """
        self.index = index
        self.top_k = top_k
        self.max_excerpt_length = max_excerpt_length
        self._last_results: list[dict] = []

    def __call__(self, query: str) -> str:
        """Execute search and return formatted results.

        Args:
            query: Search query string

        Returns:
            Formatted string with search results
        """
        return self.run(query)

    def run(self, query: str) -> str:
        """Execute search and return formatted results.

        Args:
            query: Search query string

        Returns:
            Formatted string with search results
        """
        if not query or not query.strip():
            self._last_results = []
            return "Error: Empty query. Please provide search terms."

        results = self.index.search(query, top_k=self.top_k)
        self._last_results = results

        if not results:
            return f"No relevant federal laws found for: '{query}'"

        formatted = []
        for doc in results:
            citation = doc.get("citation", "Unknown")
            text = doc.get("text", "")
            doc_type_label = doc.get("_type", "")
            if CONFIG.get("cch_enabled", False) and not doc_type_label:
                m = re.search(r"\b([A-Z]{2,}[a-z]?)\s*$", citation.strip())
                if m:
                    doc_type_label = m.group(1)

            # Truncate text for readability
            if len(text) > self.max_excerpt_length:
                text = text[: self.max_excerpt_length] + "..."

            if CONFIG.get("cch_enabled", False) and doc_type_label:
                formatted.append(f"- [{doc_type_label}] {citation}: {text}")
            else:
                formatted.append(f"- {citation}: {text}")

        return "\n".join(formatted)

    def get_last_citations(self) -> list[str]:
        """Return citations from the last search.

        Returns:
            List of citation strings from the most recent search
        """
        return [doc.get("citation", "") for doc in self._last_results if doc.get("citation")]


class CourtSearchTool:
    """Tool for searching Swiss Federal Court decisions corpus.

    Searches court decisions (BGE and docket-style citations)
    using BM25 keyword matching.
    """

    name: str = "search_courts"
    description: str = """Search Swiss Federal Court decisions by keywords.
Input: Search query string (German, French, Italian, or English)
Output: List of relevant court decision citations with excerpts

Use this tool to find relevant case law and judicial interpretations.
Example queries: "negligence standard of care", "Sorgfaltspflicht", "contract interpretation"
"""

    def __init__(
        self,
        index: BM25Index,
        top_k: int = 5,
        max_excerpt_length: int = 300,
    ):
        """Initialize court search tool.

        Args:
            index: BM25Index for court decisions corpus
            top_k: Number of results to return
            max_excerpt_length: Maximum characters for text excerpts
        """
        self.index = index
        self.top_k = top_k
        self.max_excerpt_length = max_excerpt_length
        self._last_results: list[dict] = []

    def __call__(self, query: str) -> str:
        """Execute search and return formatted results.

        Args:
            query: Search query string

        Returns:
            Formatted string with search results
        """
        return self.run(query)

    def run(self, query: str) -> str:
        """Execute search and return formatted results.

        Args:
            query: Search query string

        Returns:
            Formatted string with search results
        """
        if not query or not query.strip():
            self._last_results = []
            return "Error: Empty query. Please provide search terms."

        results = self.index.search(query, top_k=self.top_k)
        self._last_results = results

        if not results:
            return f"No relevant court decisions found for: '{query}'"

        formatted = []
        for doc in results:
            citation = doc.get("citation", "Unknown")
            text = doc.get("text", "")
            doc_type_label = doc.get("_type", "")
            if CONFIG.get("cch_enabled", False) and not doc_type_label:
                m = re.match(r"BGE\s+\d+\s+([IVX]+)", citation)
                if m:
                    doc_type_label = f"BGE_{m.group(1)}"
                else:
                    m = re.match(r"(\d+[A-Z]+)", citation)
                    if m:
                        doc_type_label = f"CASE_{m.group(1)}"

            # Truncate text for readability
            if len(text) > self.max_excerpt_length:
                text = text[: self.max_excerpt_length] + "..."

            if CONFIG.get("cch_enabled", False) and doc_type_label:
                formatted.append(f"- [{doc_type_label}] {citation}: {text}")
            else:
                formatted.append(f"- {citation}: {text}")

        return "\n".join(formatted)

    def get_last_citations(self) -> list[str]:
        """Return citations from the last search.

        Returns:
            List of citation strings from the most recent search
        """
        return [doc.get("citation", "") for doc in self._last_results if doc.get("citation")]


# Create tools
law_tool = LawSearchTool(
    index=laws_index,
    top_k=CONFIG["top_k_laws"],
    max_excerpt_length=300,
)

court_tool = CourtSearchTool(
    index=courts_index,
    top_k=CONFIG["top_k_courts"],
    max_excerpt_length=300, 
)

# Tool registry
TOOLS = {
    "search_laws": law_tool,
    "search_courts": court_tool,
}

print("Tools registered:")
for name, tool in TOOLS.items():
    print(f"  - {name}: {tool.description.split(chr(10))[0]}")

Tools registered:
  - search_laws: Search Swiss federal laws (SR/Systematische Rechtssammlung) by keywords.
  - search_courts: Search Swiss Federal Court decisions by keywords.


In [8]:
# Test tools
print("Testing law search:")
print(law_tool("Vertrag Abschluss"))

print("\nTesting court search:")
print(court_tool("Meinungsfreiheit"))

Testing law search:
- Art. 22 Abs. 1 OR: 1 Durch Vertrag kann die Verpflichtung zum Abschluss eines künftigen Vertrages begründet werden.
- Art. 23 OR: Der Vertrag ist für denjenigen unverbindlich, der sich beim Abschluss in einem wesentlichen Irrtum befunden hat.
- Art. 22 Abs. 3 VEAGOG: 3 Der Leistungsauftrag wird mittels Vertrag erteilt. Es besteht kein Rechtsanspruch auf den Abschluss eines Leistungsauftrags.52
- Art. 20 Abs. 2 VEAGOG: 2 Der Leistungsauftrag wird mittels Vertrag erteilt. Es besteht kein Rechtsanspruch auf den Abschluss eines Leistungsauftrags.47
- Art. 166 Abs. 2 BV: 2 Sie genehmigt die völkerrechtlichen Verträge; ausgenommen sind die Verträge, für deren Abschluss auf Grund von Gesetz oder völkerrechtlichem Vertrag der Bundesrat zuständig ist.
- Art. 152 Abs. 3bis ParlG: 3bis Der Bundesrat konsultiert die zuständigen Kommissionen, bevor er:a. einen völkerrechtlichen Vertrag vorläufig anwendet, dessen Abschluss oder Änderung durch die Bundesversammlung genehmigt wer

## 4. Load Local LLM

In [9]:
from llama_cpp import Llama

# Find model file
model_file = MODEL_PATH / CONFIG["model_file"]
if not model_file.exists():
    gguf_files = list(MODEL_PATH.glob("*.gguf")) + list(MODEL_PATH.rglob("*.gguf"))
    if gguf_files:
        model_file = gguf_files[0]
    else:
        raise FileNotFoundError(f"No GGUF model found in {MODEL_PATH}")

print(f"Loading model: {model_file}")
llm = Llama(
    model_path=str(model_file),
    n_ctx=CONFIG["n_ctx"],
    n_threads=CONFIG["n_threads"],
    n_gpu_layers=CONFIG["n_gpu_layers"],
    verbose=False,
)
print(f"Model loaded on GPU (all layers offloaded)")


Loading model: /kaggle/input/datasets/charan1996/mistral-7b-gguf/mistral-7b-instruct-v0.2.Q4_K_M.gguf
Model loaded on GPU (all layers offloaded)


## 5. Build Few-Shot Example Bank (Hybrid: train.csv + Corpus Mining)

**Hybrid approach** for exhaustive type coverage with **3 examples per type**:
1. **Real pairs from train.csv** (priority): Match gold citations to corpus text → real (query, passage) pairs per type. Keep up to 3 best (shortest query = most focused).
2. **Synthetic fill**: For types with < 3 examples from train.csv, generate synthetic queries via Mistral to fill up to 3 per type.
3. **English translations**: Each example gets an English translation of the query, used later for **domain-matched few-shot selection** (matching input query to relevant type examples without LLM routing).

This ensures every type has 3 few-shot examples, and the English queries enable keyword-based routing to select domain-matched examples for each input query at inference time.

In [10]:
import pandas as pd
from collections import defaultdict
from tqdm.notebook import tqdm

# ============================================================
# FEW-SHOT BANK SETUP (conditional)
# ============================================================
# Keep this cell cheap in baseline mode:
# - If few_shot_enabled=False, do not build or load few-shot banks.
# - If few_shot_enabled=True, load cache or build from train/corpus.
# ============================================================

EXAMPLES_PER_TYPE = CONFIG.get("hyde_examples_per_type", 3)
FEW_SHOT_CACHE_PATH = INDEX_PATH / "few_shot_banks.pkl"


def get_law_type(citation):
    """Extract statute abbreviation: 'Art. 10a Abs. 1 USG' -> 'USG'."""
    match = re.search(r"\b([A-Z]{2,}[a-z]?)\s*$", citation.strip())
    if match:
        return match.group(1)
    matches = re.findall(r"\b([A-Z]{2,})\b", citation)
    return matches[-1] if matches else "OTHER"


def get_court_type(citation):
    """Extract court type: 'BGE 142 III ...' -> 'BGE_III', '1C_...' -> 'CASE_1C'."""
    m = re.match(r"BGE\s+\d+\s+([IVX]+)", citation)
    if m:
        return f"BGE_{m.group(1)}"
    m = re.match(r"(\d+[A-Z]+)", citation)
    if m:
        return f"CASE_{m.group(1)}"
    return "OTHER"


# Always define these for downstream cells.
law_few_shot_bank = {}
court_few_shot_bank = {}
selected_law_examples = []
selected_court_examples = []

if not CONFIG.get("few_shot_enabled", False):
    print("Few-shot disabled: skipping bank load/build for baseline-style run.")
else:
    if FEW_SHOT_CACHE_PATH.exists() and not FORCE_REBUILD_INDICES:
        print(f"Loading cached few-shot banks from {FEW_SHOT_CACHE_PATH}")
        with open(FEW_SHOT_CACHE_PATH, "rb") as f:
            cache_data = pickle.load(f)

        law_few_shot_bank = cache_data["law_few_shot_bank"]
        court_few_shot_bank = cache_data["court_few_shot_bank"]
        selected_law_examples = cache_data["selected_law_examples"]
        selected_court_examples = cache_data["selected_court_examples"]

        real_law_count = sum(1 for e in selected_law_examples if e["source"] == "train.csv")
        syn_law_count = sum(1 for e in selected_law_examples if e["source"] == "synthetic")
        real_court_count = sum(1 for e in selected_court_examples if e["source"] == "train.csv")
        syn_court_count = sum(1 for e in selected_court_examples if e["source"] == "synthetic")

        print(
            f"  Laws:   {len(law_few_shot_bank)} types, {len(selected_law_examples)} examples "
            f"({real_law_count} real + {syn_law_count} synthetic)"
        )
        print(
            f"  Courts: {len(court_few_shot_bank)} types, {len(selected_court_examples)} examples "
            f"({real_court_count} real + {syn_court_count} synthetic)"
        )
        print("  Examples use German queries for matching")
    else:
        print("Building few-shot banks from scratch (will cache for future runs)...")

        print("\nBuilding citation -> text lookups from loaded indices...")
        law_citation_to_text = {}
        court_citation_to_text = {}

        for doc in laws_index.documents:
            cit = doc.get("citation", "")
            text = doc.get("text", "")
            if cit and text:
                law_citation_to_text[cit] = text

        for doc in courts_index.documents:
            cit = doc.get("citation", "")
            text = doc.get("text", "")
            if cit and text:
                court_citation_to_text[cit] = text

        print(f"  Laws entries: {len(law_citation_to_text)}")
        print(f"  Courts entries: {len(court_citation_to_text)}")

        corpus_law_types = defaultdict(list)
        for cit, text in law_citation_to_text.items():
            t = get_law_type(cit)
            if len(corpus_law_types[t]) < 10:
                corpus_law_types[t].append({"citation": cit, "text": text[:600]})

        corpus_court_types = defaultdict(list)
        for cit, text in court_citation_to_text.items():
            t = get_court_type(cit)
            if len(corpus_court_types[t]) < 10:
                corpus_court_types[t].append({"citation": cit, "text": text[:600]})

        print("\nCorpus type coverage:")
        print(f"  Law types in index: {len(corpus_law_types)}")
        print(f"  Court types in index: {len(corpus_court_types)}")

        train_df = pd.read_csv(DATA_PATH / "train.csv")
        print(f"\nLoaded {len(train_df)} training examples from train.csv")

        real_law_examples = defaultdict(list)
        real_court_examples = defaultdict(list)

        for _, row in train_df.iterrows():
            query = str(row["query"])
            gold = str(row.get("gold_citations", ""))
            if not gold or gold == "nan":
                continue
            for cit in [c.strip() for c in gold.split(";") if c.strip()]:
                if cit in law_citation_to_text:
                    t = get_law_type(cit)
                    existing_cits = {e["citation"] for e in real_law_examples[t]}
                    if cit not in existing_cits:
                        real_law_examples[t].append(
                            {
                                "query": query[:500],
                                "citation": cit,
                                "text": law_citation_to_text[cit][:600],
                                "source": "train.csv",
                            }
                        )
                elif cit in court_citation_to_text:
                    t = get_court_type(cit)
                    existing_cits = {e["citation"] for e in real_court_examples[t]}
                    if cit not in existing_cits:
                        real_court_examples[t].append(
                            {
                                "query": query[:500],
                                "citation": cit,
                                "text": court_citation_to_text[cit][:600],
                                "source": "train.csv",
                            }
                        )

        for t in real_law_examples:
            real_law_examples[t] = sorted(real_law_examples[t], key=lambda e: len(e["query"]))[:EXAMPLES_PER_TYPE]
        for t in real_court_examples:
            real_court_examples[t] = sorted(real_court_examples[t], key=lambda e: len(e["query"]))[:EXAMPLES_PER_TYPE]

        MAX_SYNTHETIC_TYPES = CONFIG.get("hyde_max_synthetic_types", 50)

        def generate_synthetic_query(text_snippet, doc_type="law"):
            """Use Mistral to generate a synthetic German query for a text snippet."""
            task = "Schweizer Gesetzesartikel" if doc_type == "law" else "Schweizer Gerichtserwaegung"
            prompt = (
                f"[INST] Gegeben der folgende {task}, schreibe eine kurze rechtliche Frage auf Deutsch, "
                f"die dieser Text beantworten koennte. Nur die Frage, keine Erklaerung.\n\n"
                f"Text: {text_snippet[:400]}\n\n"
                f"Frage: [/INST]"
            )
            try:
                response = llm(
                    prompt,
                    max_tokens=100,
                    temperature=0.3,
                    stop=["[INST]", "</s>", "\n\n"],
                )
                return response["choices"][0]["text"].strip()
            except Exception:
                return f"Rechtliche Frage zu diesem {task}?"

        law_types_needing_fill = []
        for t in corpus_law_types:
            current_count = len(real_law_examples.get(t, []))
            if current_count < EXAMPLES_PER_TYPE:
                law_types_needing_fill.append((t, EXAMPLES_PER_TYPE - current_count))

        court_types_needing_fill = []
        for t in corpus_court_types:
            current_count = len(real_court_examples.get(t, []))
            if current_count < EXAMPLES_PER_TYPE:
                court_types_needing_fill.append((t, EXAMPLES_PER_TYPE - current_count))

        law_types_needing_fill.sort(key=lambda x: -len(corpus_law_types.get(x[0], [])))
        law_types_needing_fill = law_types_needing_fill[:MAX_SYNTHETIC_TYPES]

        court_types_needing_fill.sort(key=lambda x: -len(corpus_court_types.get(x[0], [])))
        court_types_needing_fill = court_types_needing_fill[:MAX_SYNTHETIC_TYPES]

        print("\nGenerating synthetic queries for law types...")
        for t, needed in tqdm(law_types_needing_fill, desc="Synthetic law queries"):
            candidates = corpus_law_types[t]
            existing_cits = {e["citation"] for e in real_law_examples.get(t, [])}
            available = sorted(
                [c for c in candidates if c["citation"] not in existing_cits],
                key=lambda c: -len(c["text"]),
            )
            for j in range(min(needed, len(available))):
                syn_query = generate_synthetic_query(available[j]["text"], doc_type="law")
                real_law_examples[t].append(
                    {
                        "query": syn_query[:500],
                        "citation": available[j]["citation"],
                        "text": available[j]["text"],
                        "source": "synthetic",
                    }
                )

        print("\nGenerating synthetic queries for court types...")
        for t, needed in tqdm(court_types_needing_fill, desc="Synthetic court queries"):
            candidates = corpus_court_types[t]
            existing_cits = {e["citation"] for e in real_court_examples.get(t, [])}
            available = sorted(
                [c for c in candidates if c["citation"] not in existing_cits],
                key=lambda c: -len(c["text"]),
            )
            for j in range(min(needed, len(available))):
                syn_query = generate_synthetic_query(available[j]["text"], doc_type="court")
                real_court_examples[t].append(
                    {
                        "query": syn_query[:500],
                        "citation": available[j]["citation"],
                        "text": available[j]["text"],
                        "source": "synthetic",
                    }
                )

        law_few_shot_bank = {t: exs[:EXAMPLES_PER_TYPE] for t, exs in real_law_examples.items() if exs}
        court_few_shot_bank = {t: exs[:EXAMPLES_PER_TYPE] for t, exs in real_court_examples.items() if exs}

        selected_law_examples = []
        for t in sorted(law_few_shot_bank.keys()):
            selected_law_examples.extend(law_few_shot_bank[t])

        selected_court_examples = []
        for t in sorted(court_few_shot_bank.keys()):
            selected_court_examples.extend(court_few_shot_bank[t])

        if not KAGGLE_ENV:
            print(f"\nSaving few-shot banks to {FEW_SHOT_CACHE_PATH}...")
            cache_data = {
                "law_few_shot_bank": law_few_shot_bank,
                "court_few_shot_bank": court_few_shot_bank,
                "selected_law_examples": selected_law_examples,
                "selected_court_examples": selected_court_examples,
            }
            with open(FEW_SHOT_CACHE_PATH, "wb") as f:
                pickle.dump(cache_data, f)
            print("  Cached for future runs.")

        real_law_count = sum(1 for e in selected_law_examples if e["source"] == "train.csv")
        syn_law_count = sum(1 for e in selected_law_examples if e["source"] == "synthetic")
        real_court_count = sum(1 for e in selected_court_examples if e["source"] == "train.csv")
        syn_court_count = sum(1 for e in selected_court_examples if e["source"] == "synthetic")

        print(f"\n{'=' * 60}")
        print("FINAL FEW-SHOT BANK (conditional mode):")
        print(
            f"  Laws:   {len(law_few_shot_bank)} types, {len(selected_law_examples)} examples "
            f"({real_law_count} real + {syn_law_count} synthetic)"
        )
        print(
            f"  Courts: {len(court_few_shot_bank)} types, {len(selected_court_examples)} examples "
            f"({real_court_count} real + {syn_court_count} synthetic)"
        )
        print(f"  Examples per type: up to {EXAMPLES_PER_TYPE}")
        print(f"{'=' * 60}")

if CONFIG.get("few_shot_enabled", False):
    print("\nSample law examples (OR type):")
    if "OR" in law_few_shot_bank:
        for ex in law_few_shot_bank["OR"]:
            print(f"  [{ex['source']:9s}] Q_de: {ex['query'][:60]}...")
            print()

    print("Sample court examples (BGE_III type):")
    if "BGE_III" in court_few_shot_bank:
        for ex in court_few_shot_bank["BGE_III"]:
            print(f"  [{ex['source']:9s}] Q_de: {ex['query'][:60]}...")
            print()

Few-shot disabled: skipping bank load/build for baseline-style run.


## 5b. Type Registry & Hierarchical Search Infrastructure

Adapts two proven RAG techniques ([Hierarchical Indices](https://github.com/NirDiamant/RAG_TECHNIQUES/blob/main/all_rag_techniques/hierarchical_indices.ipynb) + [Contextual Chunk Headers](https://github.com/NirDiamant/RAG_TECHNIQUES/blob/main/all_rag_techniques/contextual_chunk_headers.ipynb)) for our BM25 Swiss legal pipeline:

1. **Hierarchical Indices**: The Swiss legal corpus has natural hierarchy (Type → Article → Text). Type detection is done **upstream** by keyword matching in `select_few_shot_examples()`. This function receives `type_hints` and applies **soft-boost** to matching documents' BM25 scores.
2. **Contextual Chunk Headers (CCH)**: Type metadata is prepended to search results (`[OR] Art. 1 OR: ...`) so the LLM understands document provenance.

**Key design: Soft boost, not hard filter.** Wrong type guess → no damage. Right guess → precision improves.
**Type routing**: No LLM routing, no regex detection — types are identified by keyword overlap against the few-shot bank's `query_en` fields.

In [11]:
import numpy as np
from collections import Counter

# ============================================================
# TYPE REGISTRY & HIERARCHICAL SEARCH (conditional prep)
# ============================================================
# In notebook-2-like baseline runs, avoid extra preprocessing work.
# We only build doc_types/registries when they are needed by:
# - type_boost_enabled, or
# - prompt_injection_enabled, or
# - cch_enabled
# ============================================================

NEED_TYPE_METADATA = (
    CONFIG.get("type_boost_enabled", False)
    or CONFIG.get("prompt_injection_enabled", False)
    or CONFIG.get("cch_enabled", False)
)

law_type_counts = Counter()
court_type_counts = Counter()
LAW_TYPE_REGISTRY = {}
COURT_TYPE_REGISTRY = {}
LAW_TYPES_FOR_PROMPT = ""
COURT_TYPES_FOR_PROMPT = ""

if NEED_TYPE_METADATA:
    print("Computing doc_types arrays for hierarchical search...")
    laws_index.doc_types = np.array(
        [get_law_type(doc["citation"]) for doc in laws_index.documents],
        dtype=object,
    )
    courts_index.doc_types = np.array(
        [get_court_type(doc["citation"]) for doc in courts_index.documents],
        dtype=object,
    )

    law_type_counts = Counter(laws_index.doc_types)
    court_type_counts = Counter(courts_index.doc_types)

    print(f"  Laws:   {len(laws_index.doc_types):,} docs -> {len(law_type_counts)} unique types")
    print(f"  Courts: {len(courts_index.doc_types):,} docs -> {len(court_type_counts)} unique types")

    def _build_registry(type_counts, index):
        """Build registry with count and example citation per type."""
        registry = {}
        seen = set()
        for i, doc in enumerate(index.documents):
            t = index.doc_types[i]
            if t not in seen:
                registry[t] = {"count": int(type_counts[t]), "example": doc["citation"]}
                seen.add(t)
            if len(seen) == len(type_counts):
                break
        return registry

    LAW_TYPE_REGISTRY = _build_registry(law_type_counts, laws_index)
    COURT_TYPE_REGISTRY = _build_registry(court_type_counts, courts_index)

    _top_laws = sorted(LAW_TYPE_REGISTRY.items(), key=lambda x: -x[1]["count"])
    LAW_TYPES_FOR_PROMPT = ", ".join(
        f"{t}({info['count']})" for t, info in _top_laws[:40]
    )
    COURT_TYPES_FOR_PROMPT = ", ".join(
        f"{t}({info['count']})" for t, info in
        sorted(COURT_TYPE_REGISTRY.items(), key=lambda x: -x[1]["count"])
    )

    print(f"\nTYPE_REGISTRY: {len(LAW_TYPE_REGISTRY)} law + {len(COURT_TYPE_REGISTRY)} court types")
else:
    print("Skipping type metadata build (baseline-style run).")


def hierarchical_bm25_search(index, query, top_k=10, type_hints=None, boost_factor=None):
    """Two-level BM25 search with optional type-aware score boosting."""
    if boost_factor is None:
        boost_factor = CONFIG.get("type_boost_factor", 1.5)
    if type_hints is None:
        type_hints = []

    query_tokens = index.tokenize(query)
    if not query_tokens:
        return [], None

    scores = index.index.get_scores(query_tokens)
    type_info = None

    if type_hints and hasattr(index, "doc_types"):
        boost_mask = np.where(
            np.isin(index.doc_types, type_hints),
            boost_factor,
            1.0,
        )
        scores = scores * boost_mask
        type_info = {"types": list(type_hints), "source": "keyword_matching"}

    top_indices = scores.argsort()[-top_k:][::-1]
    results = []
    for idx in top_indices:
        if scores[idx] <= 0:
            continue
        doc = index.documents[idx].copy()
        doc["_score"] = float(scores[idx])
        if hasattr(index, "doc_types"):
            doc["_type"] = str(index.doc_types[idx])
        results.append(doc)

    return results, type_info

print(f"\n{'=' * 60}")
print("HIERARCHICAL SEARCH INFRASTRUCTURE READY")
print(f"  Need type metadata: {NEED_TYPE_METADATA}")
print(f"  Type boost enabled: {CONFIG['type_boost_enabled']}")
print(f"  Prompt injection enabled: {CONFIG['prompt_injection_enabled']}")
print(f"  CCH enabled: {CONFIG['cch_enabled']}")
if NEED_TYPE_METADATA:
    print(f"  Law types for prompt: top 40 of {len(LAW_TYPE_REGISTRY)}")
    print(f"  Court types for prompt: all {len(COURT_TYPE_REGISTRY)}")
print(f"{'=' * 60}")

Skipping type metadata build (baseline-style run).

HIERARCHICAL SEARCH INFRASTRUCTURE READY
  Need type metadata: False
  Type boost enabled: False
  Prompt injection enabled: False
  CCH enabled: False


## 6. HyDE — Hypothetical Document Generation & Enhanced Search Tools

The HyDE pipeline now works as follows:
1. **Keyword matching** against the few-shot bank's `query_en` → detects relevant type(s) + selects matched examples
2. **HyDE generation**: Creates a hypothetical German legal passage using the matched few-shot examples
3. **Single BM25 search** with the hypothetical document + type boost from keyword matching
4. **CCH formatting**: Results labelled with `[type]` for LLM provenance context

**No LLM routing**: The agent sends German queries. Type detection is entirely keyword-based.
**No dual search**: Only the hypothetical document is searched (no separate keyword search).
**Type boost**: Applied from the `type_hints` produced by keyword matching — not from query regex or dominant-type auto-detection.

In [12]:
import hashlib
from collections import defaultdict

# ============================================================
# Keyword-Based Few-Shot Selection + Type Detection
# ============================================================
# Instead of LLM-driven routing, we detect relevant types by scoring
# each bank example's query against the input German query.
# This gives us BOTH the matched examples AND the type_hints in one step.

# Common English stop words (module-level constant, not recreated per call)
_STOP_WORDS = frozenset({
    "the", "a", "an", "is", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "do", "does", "did", "will", "would", "shall",
    "should", "may", "might", "must", "can", "could", "to", "of", "in",
    "for", "on", "with", "at", "by", "from", "as", "into", "through",
    "during", "before", "after", "above", "below", "between", "under",
    "and", "but", "or", "nor", "not", "no", "so", "if", "then", "than",
    "too", "very", "just", "about", "up", "out", "that", "this", "it",
    "its", "what", "which", "who", "whom", "how", "when", "where", "why",
    "über", "auch", "oder", "aber", "nicht", "noch", "schon",
    "sein", "haben", "werden", "können", "müssen", "sollen",
    "bei", "nach", "vor", "zwischen", "gegen", "ohne", "seit",
    "des", "dem", "den", "einer", "einem", "eines",
})

def select_few_shot_examples(query, doc_type="law", n=None):
    """Select few-shot examples via keyword overlap AND detect type_hints.
    
    Strategy:
    1. Score every example's query against the input German query
       by counting shared words (case-insensitive, stop-words removed).
    2. Group scores by type → compute per-type aggregate score.
    3. Determine type_hints:
       - If top type's aggregate >= 1.5x the runner-up → single dominant type
       - If top two types are close → both are relevant
       - If no meaningful overlap → type_hints = []
    4. Select examples coordinated with type_hints.
    
    Args:
        query: Input query in English
        doc_type: "law" or "court"
        n: Number of examples to return (default from CONFIG)
    
    Returns:
        Tuple of (examples_list, type_hints_list)
    """
    if n is None:
        n = CONFIG.get("hyde_few_shot_count", 3)
    
    bank = law_few_shot_bank if doc_type == "law" else court_few_shot_bank
    
    if not bank:
        return [], []
    
    query_words = set(query.lower().split()) - _STOP_WORDS
    
    if not query_words:
        # Query is all stop words — fallback to first N from largest types
        all_flat = []
        for t in sorted(bank.keys(), key=lambda t: -len(bank[t])):
            all_flat.extend(bank[t])
        return all_flat[:n], []
    
    # Score each example by word overlap with query
    scored_examples = []  # [(overlap_score, type, example_dict)]
    for t, examples in bank.items():
        for ex in examples:
            ex_text = ex.get("query", "").lower()
            ex_words = set(ex_text.split()) - _STOP_WORDS
            overlap = len(query_words & ex_words)
            scored_examples.append((overlap, t, ex))
    
    # Aggregate per-type scores
    type_scores = defaultdict(float)
    type_max_scores = defaultdict(float)
    for overlap, t, _ in scored_examples:
        type_scores[t] += overlap
        type_max_scores[t] = max(type_max_scores[t], overlap)
    
    # Rank types with any overlap
    ranked_types = sorted(
        [t for t in type_scores if type_scores[t] > 0],
        key=lambda t: (type_scores[t], type_max_scores[t]),
        reverse=True
    )
    
    if not ranked_types:
        all_flat = []
        for t in sorted(bank.keys(), key=lambda t: -len(bank[t])):
            all_flat.extend(bank[t])
        return all_flat[:n], []
    
    # Determine type_hints
    top_score = type_scores[ranked_types[0]]
    second_score = type_scores[ranked_types[1]] if len(ranked_types) >= 2 else 0
    DOMINANCE_RATIO = 1.5
    
    if second_score > 0 and top_score < DOMINANCE_RATIO * second_score:
        type_hints = [ranked_types[0], ranked_types[1]]
    else:
        type_hints = [ranked_types[0]]
    
    # Select examples coordinated with type_hints
    if len(type_hints) == 1:
        result = list(bank[type_hints[0]][:n])
        if len(result) < n and len(ranked_types) >= 2:
            for ex in bank[ranked_types[1]]:
                if ex not in result:
                    result.append(ex)
                    if len(result) >= n:
                        break
    else:
        score1 = type_scores[type_hints[0]]
        score2 = type_scores[type_hints[1]]
        total = score1 + score2
        from_top = max(1, min(n - 1, round(n * score1 / total)))
        from_second = n - from_top
        result = list(bank[type_hints[0]][:from_top])
        result.extend(bank[type_hints[1]][:from_second])
    
    # Fill remaining slots from top individually-scored examples
    if len(result) < n:
        scored_examples.sort(key=lambda x: -x[0])
        for _, _, ex in scored_examples:
            if ex not in result:
                result.append(ex)
                if len(result) >= n:
                    break
    
    return result[:n], type_hints


# ============================================================
# HyDE Prompt Builder (v2 — cross-lingual bridge)
# ============================================================
# Shows the LLM explicit English→German mapping in few-shot examples:
#   English question → German question → German legal text
# This teaches the model the cross-lingual bridge pattern.

def build_hyde_prompt(query, doc_type="law", few_shot_examples=None):
    """Build a HyDE prompt with cross-lingual few-shot examples.
    
    The few-shot examples show:
      English question (query_en) → German question (query) → German legal text
    This teaches Mistral the English→German generation pattern explicitly.
    
    Args:
        query: The search query (English — will instruct LLM to write German)
        doc_type: "law" for Gesetzestext, "court" for Erwägung
        few_shot_examples: List of {query, query_en, citation, text} dicts
    
    Returns:
        Formatted prompt string for Mistral Instruct
    """
    if doc_type == "law":
        instruction = (
            "Du bist ein Schweizer Rechtsexperte. Gegeben eine rechtliche Frage, "
            "schreibe einen hypothetischen Schweizer Gesetzesartikel auf Deutsch, "
            "der diese Frage beantworten würde.\n"
            "Schreibe den Text wie einen echten Gesetzesartikel (Gesetzestext), "
            "nicht als Antwort oder Erklärung.\n"
            f"Der Text soll ca. {CONFIG['hyde_target_chars_law']} Zeichen lang sein.\n"
            "Wenn die Frage auf Englisch ist, schreibe trotzdem auf Deutsch."
        )
    else:
        instruction = (
            "Du bist ein Schweizer Rechtsexperte. Gegeben eine rechtliche Frage, "
            "schreibe eine hypothetische Erwägung eines Schweizer Bundesgerichtsentscheids "
            "auf Deutsch, die diese Frage behandeln würde.\n"
            "Schreibe den Text wie eine echte Gerichtserwägung (BGE), "
            "nicht als Antwort oder Erklärung.\n"
            f"Der Text soll ca. {CONFIG['hyde_target_chars_court']} Zeichen lang sein.\n"
            "Wenn die Frage auf Englisch ist, schreibe trotzdem auf Deutsch."
        )
    
    # Add few-shot examples with cross-lingual bridge:
    # English query → German query → German hypothetical text
    examples_text = ""
    if few_shot_examples:
        n = CONFIG.get("hyde_few_shot_count", 3)
        examples_text = "\n\nBeispiele:\n"
        for ex in few_shot_examples[:n]:
            query_de = ex.get("query", "")
            # With German agent, just show German query + German text
            examples_text += "\nFrage: " + query_de[:300] + "\n"
            examples_text += "Hypothetischer Text: " + ex["text"][:400] + "\n"

    
    prompt = f"[INST] {instruction}{examples_text}\n\nFrage: {query}\n\nHypothetischer Text: [/INST]"
    return prompt


# ============================================================
# HyDE Document Generator  
# ============================================================

_hyde_cache = {}  # Cache: hash(query+type) → generated text

def generate_hypothetical_document(query, doc_type="law", few_shot_examples=None):
    """Generate a hypothetical German legal document for HyDE retrieval.
    
    Args:
        query: Search query (English)
        doc_type: "law" or "court"
        few_shot_examples: List of {query, query_en, citation, text} dicts
    
    Returns:
        Generated hypothetical document text (German)
    """
    if not CONFIG.get("hyde_enabled", True):
        return query  # Bypass: return original query for ablation
    
    # Check cache
    cache_key = hashlib.md5(f"{query}:{doc_type}".encode()).hexdigest()
    if cache_key in _hyde_cache:
        return _hyde_cache[cache_key]
    
    prompt = build_hyde_prompt(query, doc_type, few_shot_examples)
    
    try:
        response = llm(
            prompt,
            max_tokens=CONFIG.get("hyde_max_tokens", 300),
            temperature=CONFIG.get("hyde_temperature", 0.3),
            stop=["[INST]", "</s>", "\nFrage:", "\n\nFrage:"],
        )
        hyde_doc = response["choices"][0]["text"].strip()
    except Exception as e:
        print(f"  [HyDE] Generation failed ({doc_type}): {e}")
        hyde_doc = query  # Fallback to original query
    
    _hyde_cache[cache_key] = hyde_doc
    return hyde_doc


# ============================================================
# HyDE + Hierarchical Search Tools
# ============================================================
# Pipeline per tool call:
#   1. Keyword-match query against few-shot bank → get examples + type_hints
#   2. Build HyDE prompt with matched examples → generate German hypothetical doc
#   3. Single BM25 search with hypothetical doc + type boost from type_hints
#   4. Format results with CCH-style [type] labels

class HyDELawSearchTool:
    """HyDE + hierarchical law search: keyword-matches for type detection,
    generates hypothetical Gesetzestext, then BM25-searches with type boosting."""
    
    name: str = "search_laws"
    description: str = (
        "Search Swiss federal laws (SR/Systematische Rechtssammlung) by keywords.\n"
        "Input: Search query string in English\n"
        "Output: List of relevant law citations with text excerpts\n"
        "TIP: Be specific about the legal area (contracts, criminal, family, etc)."
    )

    def __init__(self, index, top_k=5, max_excerpt_length=300):
        self.index = index
        self.top_k = top_k
        self.max_excerpt_length = max_excerpt_length
        self._last_results = []

    def __call__(self, query):
        return self.run(query)

    def run(self, query):
        if not query or not query.strip():
            self._last_results = []
            return "Error: Empty query. Please provide search terms."
        # Step 1: Few-shot matching (skip if disabled)
        if CONFIG.get("few_shot_enabled", True):
            matched_examples, type_hints = select_few_shot_examples(query, doc_type="law")
        else:
            matched_examples, type_hints = [], []

        # Step 2: Type boost guard
        if not CONFIG.get("type_boost_enabled", True):
            type_hints = []

        # Step 3: Generate hypothetical German law article (HyDE)
        hyde_doc = generate_hypothetical_document(
            query, doc_type="law", few_shot_examples=matched_examples
        )

        # Step 4: BM25 search with hypothetical doc + type boost
        results, type_info = hierarchical_bm25_search(
            self.index, hyde_doc, top_k=self.top_k, type_hints=type_hints
        )
        
        self._last_results = results
        
        if not self._last_results:
            return f"No relevant federal laws found for: '{query}'"
        
        # Step 4: Format with CCH-style type labels
        formatted = []
        for doc in results:
            citation = doc.get("citation", "Unknown")
            text = doc.get("text", "")
            doc_type_label = doc.get("_type", "")
            if len(text) > self.max_excerpt_length:
                text = text[:self.max_excerpt_length] + "..."
            formatted.append(f"- [{doc_type_label}] {citation}: {text}")
        
        # Type routing transparency header
        header = ""
        if CONFIG.get("cch_enabled", False) and type_info:
            types_str = ", ".join(type_info["types"])
            header = f"[Type boost: {types_str} (keyword matching)]\n"
        
        return header + "\n".join(formatted)

    def get_last_citations(self):
        return [doc.get("citation", "") for doc in self._last_results if doc.get("citation")]


class HyDECourtSearchTool:
    """HyDE + hierarchical court search: keyword-matches for type detection,
    generates hypothetical Erwägung, then BM25-searches with type boosting."""
    
    name: str = "search_courts"
    description: str = (
        "Search Swiss Federal Court decisions by keywords.\n"
        "Input: Search query string in English\n"
        "Output: List of relevant court decision citations with excerpts\n"
        "TIP: Be specific about the legal area and issue."
    )

    def __init__(self, index, top_k=5, max_excerpt_length=300):
        self.index = index
        self.top_k = top_k
        self.max_excerpt_length = max_excerpt_length
        self._last_results = []

    def __call__(self, query):
        return self.run(query)

    def run(self, query):
        if not query or not query.strip():
            self._last_results = []
            return "Error: Empty query. Please provide search terms."
        
        # Step 1: Few-shot matching (skip if disabled)
        if CONFIG.get("few_shot_enabled", True):
            matched_examples, type_hints = select_few_shot_examples(query, doc_type="court")
        else:
            matched_examples, type_hints = [], []

        # Step 2: Type boost guard
        if not CONFIG.get("type_boost_enabled", True):
            type_hints = []

        # Step 3: Generate hypothetical court consideration (HyDE)
        hyde_doc = generate_hypothetical_document(
            query, doc_type="court", few_shot_examples=matched_examples
        )

        # Step 4: BM25 search with hypothetical doc + type boost
        results, type_info = hierarchical_bm25_search(
            self.index, hyde_doc, top_k=self.top_k, type_hints=type_hints
        )
        
        self._last_results = results
        
        if not self._last_results:
            return f"No relevant court decisions found for: '{query}'"
        
        # Step 4: Format with CCH-style type labels
        formatted = []
        for doc in results:
            citation = doc.get("citation", "Unknown")
            text = doc.get("text", "")
            doc_type_label = doc.get("_type", "")
            if len(text) > self.max_excerpt_length:
                text = text[:self.max_excerpt_length] + "..."
            formatted.append(f"- [{doc_type_label}] {citation}: {text}")
        
        # Type routing transparency header
        header = ""
        if CONFIG.get("cch_enabled", False) and type_info:
            types_str = ", ".join(type_info["types"])
            header = f"[Type boost: {types_str} (keyword matching)]\n"
        
        return header + "\n".join(formatted)

    def get_last_citations(self):
        return [doc.get("citation", "") for doc in self._last_results if doc.get("citation")]


# ============================================================
# Tool Selection: strict baseline parity vs enhanced HyDE stack
# ============================================================
BASELINE_PARITY_MODE = (
    not CONFIG.get("hyde_enabled", False)
    and not CONFIG.get("few_shot_enabled", False)
    and not CONFIG.get("type_boost_enabled", False)
)

if BASELINE_PARITY_MODE:
    # Use exact notebook-02 tool behavior when retrieval features are off.
    law_tool = LawSearchTool(
        index=laws_index,
        top_k=CONFIG["top_k_laws"],
        max_excerpt_length=300,
    )

    court_tool = CourtSearchTool(
        index=courts_index,
        top_k=CONFIG["top_k_courts"],
        max_excerpt_length=300,
    )

    TOOLS = {
        "search_laws": law_tool,
        "search_courts": court_tool,
    }

    print("Baseline tools registered (Notebook-02 parity mode):")
    for name, tool in TOOLS.items():
        print(f"  - {name}: {tool.__class__.__name__}")
else:
    law_tool = HyDELawSearchTool(
        index=laws_index,
        top_k=CONFIG["top_k_laws"],
        max_excerpt_length=300,
    )

    court_tool = HyDECourtSearchTool(
        index=courts_index,
        top_k=CONFIG["top_k_courts"],
        max_excerpt_length=300,
    )

    TOOLS = {
        "search_laws": law_tool,
        "search_courts": court_tool,
    }

    print("HyDE + Hierarchical tools registered:")
    for name, tool in TOOLS.items():
        print(f"  - {name}: {tool.__class__.__name__}")

print(f"\nBaseline parity mode: {BASELINE_PARITY_MODE}")
print(f"HyDE cache size: {len(_hyde_cache)}")
print(f"HyDE enabled: {CONFIG['hyde_enabled']}")
print(f"Type boost factor: {CONFIG.get('type_boost_factor', 1.5)}")
print(f"CCH output enabled: {CONFIG.get('cch_enabled', False)}")
print(f"Type detection: keyword matching against few-shot bank query_en")

Baseline tools registered (Notebook-02 parity mode):
  - search_laws: LawSearchTool
  - search_courts: CourtSearchTool

Baseline parity mode: True
HyDE cache size: 0
HyDE enabled: False
Type boost factor: 1.5
CCH output enabled: False
Type detection: keyword matching against few-shot bank query_en


In [13]:
# === Test HyDE generation and search ===

test_query = "What are the requirements for a valid contract under Swiss law?"
print(f"Test query: {test_query}\n")

# Test keyword-based type detection + few-shot selection
print("=" * 60)
print("KEYWORD-BASED TYPE DETECTION + FEW-SHOT SELECTION:")
print("=" * 60)
matched_law_ex, law_type_hints = select_few_shot_examples(test_query, doc_type="law")
print(f"  Law type_hints: {law_type_hints}")
print(f"  Selected {len(matched_law_ex)} examples:")
for ex in matched_law_ex:
    print(f"    - [{ex.get('source', '?')}] Q_en: {ex.get('query_en', ex['query'])[:80]}...")

matched_court_ex, court_type_hints = select_few_shot_examples(test_query, doc_type="court")
print(f"\n  Court type_hints: {court_type_hints}")
print(f"  Selected {len(matched_court_ex)} examples:")
for ex in matched_court_ex:
    print(f"    - [{ex.get('source', '?')}] Q_en: {ex.get('query_en', ex['query'])[:80]}...")

# Generate hypothetical law article
print("\n" + "=" * 60)
print("HYPOTHETICAL LAW ARTICLE (Gesetzestext):")
print("=" * 60)
hyde_law = generate_hypothetical_document(
    test_query, doc_type="law", few_shot_examples=matched_law_ex
)
print(hyde_law)
print(f"\n[Length: {len(hyde_law)} chars]")

# Generate hypothetical court consideration
print("\n" + "=" * 60)
print("HYPOTHETICAL COURT CONSIDERATION (Erwägung):")
print("=" * 60)
hyde_court = generate_hypothetical_document(
    test_query, doc_type="court", few_shot_examples=matched_court_ex
)
print(hyde_court)
print(f"\n[Length: {len(hyde_court)} chars]")

# Test full tool pipeline (English query → keyword matching → HyDE → single BM25 search)
print("\n" + "=" * 60)
print("FULL TOOL PIPELINE TEST (English query in, citations out):")
print("=" * 60)

hyde_law_results = law_tool(test_query)
hyde_citations = law_tool.get_last_citations()
print(f"\nHyDE law search citations: {len(hyde_citations)}")
for c in hyde_citations[:10]:
    print(f"  - {c}")

# Baseline comparison: plain BM25 without HyDE or type boost
baseline_results = laws_index.search(test_query, top_k=CONFIG["top_k_laws"])
baseline_citations = [d.get("citation", "") for d in baseline_results if d.get("citation")]

hyde_set = set(hyde_citations)
baseline_set = set(baseline_citations)

print(f"\nComparison vs baseline BM25 (no HyDE, no boost):")
print(f"  HyDE+boost citations: {len(hyde_set)}")
print(f"  Baseline citations: {len(baseline_set)}")
print(f"  Overlap: {len(hyde_set & baseline_set)}")
print(f"  HyDE-only (new finds): {len(hyde_set - baseline_set)}")
print(f"  Baseline-only (would miss): {len(baseline_set - hyde_set)}")

if hyde_set - baseline_set:
    print(f"\nNew citations found by HyDE+boost:")
    for c in list(hyde_set - baseline_set)[:10]:
        print(f"  + {c}")

Test query: What are the requirements for a valid contract under Swiss law?

KEYWORD-BASED TYPE DETECTION + FEW-SHOT SELECTION:
  Law type_hints: []
  Selected 0 examples:

  Court type_hints: []
  Selected 0 examples:

HYPOTHETICAL LAW ARTICLE (Gesetzestext):
What are the requirements for a valid contract under Swiss law?

[Length: 63 chars]

HYPOTHETICAL COURT CONSIDERATION (Erwägung):
What are the requirements for a valid contract under Swiss law?

[Length: 63 chars]

FULL TOOL PIPELINE TEST (English query in, citations out):

HyDE law search citations: 40
  - Art. 1 Abs. 1 974.4
  - Art. 6 Abs. 2 ChKV
  - Art. 197e Abs. 2 AVO
  - Art. 9 VATT
  - Art. 3 Abs. 1 221.434
  - Art. 1 Abs. 1 VASR
  - Art. 3 Abs. 4 232.119
  - Art. 12 Abs. 1 935.911
  - Art. 2 Abs. 4 KoVo
  - Art. 48 Abs. 1 RPV

Comparison vs baseline BM25 (no HyDE, no boost):
  HyDE+boost citations: 40
  Baseline citations: 40
  Overlap: 40
  HyDE-only (new finds): 0
  Baseline-only (would miss): 0


## 7. Define ReAct Agent

In [14]:
import re

AGENT_SYSTEM_PROMPT = """Du bist ein Schweizer Rechtsrecherche-Assistent mit Zugang zu zwei Such-Tools:

1. search_laws(query): Durchsuche Schweizer Bundesgesetze (SR/Systematische Rechtssammlung)
   - Gibt relevante Gesetzesbestimmungen mit Zitaten und Textauszügen zurück
   - Verwende für Gesetzesrecht: Kodizes, Gesetze, Verordnungen

2. search_courts(query): Durchsuche Schweizer Bundesgerichtsentscheide (BGE)
   - Gibt relevante Rechtsprechung mit Zitaten und Auszügen zurück
   - Verwende für Gerichtsentscheide und Präzedenzfälle

WICHTIG: Suche IMMER auf Deutsch, da die Dokumente auf Deutsch sind.

Deine Aufgabe: Rufe die Such-Tools auf, um relevante Schweizer Rechtszitate zu finden.

Anleitung:
- Durchsuche BEIDE: Gesetze UND Gerichtsentscheide
- Verwende mehrere Suchanfragen mit deutschen Rechtsbegriffen
- Rufe die Tools auf bis alle relevanten Quellen gefunden sind

Antwortformat:
Thought: [Deine Überlegung zur nächsten Suche]
Action: [tool_name]
Action Input: [deutsche Suchanfrage]

=== BEISPIELE ===

Beispiel 1 - Vertragsrecht:
Query: What are the requirements for a valid contract?

Thought: Ich suche nach Vertragsvoraussetzungen im Obligationenrecht.
Action: search_laws
Action Input: Vertrag Abschluss Voraussetzungen OR

Observation: - Art. 1 Abs. 1 OR: Zum Abschluss eines Vertrages...

Thought: Jetzt suche ich nach BGE-Entscheiden zum Vertragsschluss.
Action: search_courts
Action Input: Vertragsabschluss Gültigkeit Voraussetzungen

Observation: - BGE 127 III 248 E. 3.1: Die Voraussetzungen...

Thought: Ich suche nach weiteren Aspekten wie Willensmängel.
Action: search_laws
Action Input: Irrtum Täuschung Vertrag OR

Beispiel 2 - Strafrecht:
Query: Under what conditions can someone be held liable for negligent homicide?

Thought: Ich suche nach fahrlässiger Tötung im Strafgesetzbuch.
Action: search_laws
Action Input: fahrlässige Tötung StGB

Observation: - Art. 117 StGB: Wer fahrlässig den Tod...

Thought: Ich suche nach Rechtsprechung zur Sorgfaltspflicht.
Action: search_courts
Action Input: fahrlässige Tötung Sorgfaltspflicht

Observation: - BGE 135 IV 56 E. 2.1: Die Sorgfaltspflicht...

Thought: Ich suche nach weiteren BGE zu Fahrlässigkeitsmassstäben.
Action: search_courts
Action Input: Fahrlässigkeit Verschulden Massstab

Beispiel 3 - Familienrecht:
Query: How is child custody determined after divorce?

Thought: Ich suche nach Sorgerecht bei Scheidung im ZGB.
Action: search_laws
Action Input: Scheidung Sorgerecht Kinder ZGB

Observation: - Art. 133 Abs. 1 ZGB: Das Gericht regelt...

Thought: Ich suche nach BGE-Entscheiden zum Kindeswohl.
Action: search_courts
Action Input: Kindeswohl Obhut Zuteilung

Observation: - BGE 142 III 481 E. 2.6: Das Kindeswohl...

Thought: Ich suche nach weiteren Bestimmungen zur elterlichen Sorge.
Action: search_laws
Action Input: elterliche Sorge Zuteilung ZGB

Beispiel 4 - Mietrecht:
Query: When can a landlord terminate a lease?

Thought: Ich suche nach Kündigungsrecht im Mietrecht.
Action: search_laws
Action Input: Mietvertrag Kündigung Vermieter OR

Observation: - Art. 266a OR: Die Kündigung ist...

Thought: Ich suche nach BGE zur missbräuchlichen Kündigung.
Action: search_courts
Action Input: Miete Kündigung missbräuchlich

Observation: - BGE 140 III 496 E. 4.1: Eine Kündigung ist...

Thought: Ich suche nach Kündigungsschutz.
Action: search_laws
Action Input: Kündigungsschutz Miete OR

=== ENDE BEISPIELE ===

Suche IMMER auf Deutsch. Rufe beide Tools (search_laws UND search_courts) auf."""


def parse_all_agent_actions(response: str) -> list[tuple[str, str]]:
    """
    Parse ALL action/input pairs from agent response.
    
    The LLM may output multiple actions in one response. This function
    extracts all of them.
    
    Args:
        response: Full LLM response text
        
    Returns:
        List of (action, action_input) tuples
    """
    actions = []
    
    # Find all "Action:" lines
    action_pattern = r"Action:\s*(\w+)"
    input_pattern = r"Action Input:\s*(.+?)(?=\nAction:|$)"
    
    # Find all action matches with their positions
    action_matches = list(re.finditer(action_pattern, response, re.IGNORECASE))
    
    for i, action_match in enumerate(action_matches):
        action = action_match.group(1).strip()
        
        # Find the corresponding Action Input
        # Start search after the Action line
        start_pos = action_match.end()
        # End search at next Action or end of string
        if i + 1 < len(action_matches):
            end_pos = action_matches[i + 1].start()
        else:
            end_pos = len(response)
        
        input_text = response[start_pos:end_pos]
        input_match = re.search(input_pattern, input_text, re.IGNORECASE | re.DOTALL)
        
        if input_match:
            action_input = input_match.group(1).strip()
            actions.append((action, action_input))
    
    return actions


def extract_citations_from_text(text: str) -> list[str]:
    """Extract citations from any text (tool output or final answer)."""
    citations = []
    
    # SR pattern: SR followed by number (optionally with article)
    sr_matches = re.findall(
        r"SR\s*\d{3}(?:\.\d+)?(?:\s+Art\.?\s*\d+[a-z]?)?",
        text,
        re.IGNORECASE
    )
    citations.extend(sr_matches)
    
    # BGE pattern: BGE volume section page
    bge_matches = re.findall(
        r"BGE\s+\d{1,3}\s+[IVX]+[a-z]?\s+\d+(?:\s+E\.\s*\d+[a-z]?)?",
        text,
        re.IGNORECASE
    )
    citations.extend(bge_matches)
    
    # Art. pattern: Art. X LAW (e.g., Art. 1 ZGB, Art. 41 OR)
    art_matches = re.findall(
        r"Art\.?\s+\d+[a-z]?\s+(?:Abs\.?\s*\d+\s+)?[A-Z]{2,}",
        text,
        re.IGNORECASE
    )
    citations.extend(art_matches)
    
    return list(set(citations))


def truncate_observation_for_llm(observation: str, max_chars: int = 1200) -> str:
    """Truncate observation text for LLM context, preserving data elsewhere."""
    if len(observation) <= max_chars:
        return observation
    return observation[:max_chars] + f"\n... (truncated, {len(observation) - max_chars} chars remaining)"


def truncate_conversation(conversation: str, max_chars: int) -> str:
    """Truncate conversation to fit within token budget, keeping system prompt and recent context."""
    if len(conversation) <= max_chars:
        return conversation
    
    inst_end = conversation.find("[/INST]")
    if inst_end == -1:
        return "..." + conversation[-max_chars:]
    
    system_part = conversation[:inst_end + 7]
    remaining_budget = max_chars - len(system_part) - 100
    
    if remaining_budget <= 0:
        return conversation[-max_chars:]
    
    rest = conversation[inst_end + 7:]
    if len(rest) > remaining_budget:
        rest = "\n...[earlier conversation truncated]...\n" + rest[-remaining_budget:]
    
    return system_part + rest


def run_agent(query: str, verbose: bool = False) -> tuple[list[str], list[dict]]:
    """Run ReAct agent to retrieve citations.
    
    Returns:
        Tuple of (citations, logs) where logs contains detailed execution information
    """
    # Format with Mistral Instruct tags
    conversation = f"[INST] {AGENT_SYSTEM_PROMPT}\n\nQuery: {query}\n\nThought: [/INST]"
    all_citations = []
    logs: list[dict] = []
    
    for iteration in range(CONFIG["max_iterations"]):
        # Truncate conversation if too long to avoid context window overflow
        max_conv_chars = CONFIG.get("max_conversation_chars", 28000)
        conversation = truncate_conversation(conversation, max_conv_chars)
        
        # Get LLM response with error handling for context overflow
        try:
            response = llm(
                conversation,
                max_tokens=CONFIG["max_tokens"],
                temperature=CONFIG["temperature"],
                stop=["Observation:", "[INST]", "</s>"],
            )["choices"][0]["text"]
        except ValueError as e:
            error_str = str(e).lower()
            if "exceed context window" in error_str or "requested tokens" in error_str:
                # Aggressively truncate and retry once
                conversation = truncate_conversation(conversation, max_chars=20000)
                try:
                    response = llm(
                        conversation,
                        max_tokens=CONFIG["max_tokens"],
                        temperature=CONFIG["temperature"],
                        stop=["Observation:", "[INST]", "</s>"],
                    )["choices"][0]["text"]
                except ValueError as retry_error:
                    # Give up, return citations found so far
                    logs.append({
                        "type": "error",
                        "iteration": iteration + 1,
                        "error": f"Context overflow after retry: {retry_error}",
                    })
                    break
            else:
                raise
        
        # For subsequent turns, we need to handle the conversation format
        if iteration == 0:
            conversation = f"[INST] {AGENT_SYSTEM_PROMPT}\n\nQuery: {query} [/INST]\n\nThought:{response}"
        else:
            conversation += response
        
        # Log LLM output
        logs.append({
            "type": "llm_response",
            "iteration": iteration + 1,
            "response": response,
            "response_trunc": response[:500] if len(response) > 500 else response,
        })
        
        if verbose:
            print(f"\n[Iteration {iteration + 1}] LLM output (trunc):")
            print(response[:500])
        
        # Parse all actions from response
        actions = parse_all_agent_actions(response)
        
        # Log parsed actions
        if actions:
            logs.append({
                "type": "parse",
                "iteration": iteration + 1,
                "actions_count": len(actions),
                "actions": actions,
            })
            if verbose:
                print(f"\n[Iteration {iteration + 1}] Parsed {len(actions)} action(s):")
                for action, action_input in actions:
                    print(f"  Action: {action}, Input: {action_input[:100]}")
        
        # Execute all actions
        observations = []
        for action, action_input in actions:
            action_lower = action.lower()
            
            if action_lower in TOOLS:
                tool = TOOLS[action_lower]
                observation = tool(action_input)
                
                # Extract citations from full observation (before truncation)
                obs_citations = tool.get_last_citations()
                all_citations.extend(obs_citations)
                
                # Truncate observation only for LLM conversation (preserve full data in logs)
                obs_truncated = truncate_observation_for_llm(observation, CONFIG["max_observation_chars"])
                observations.append(f"Tool {action_lower}: {obs_truncated}")
                
                # Log tool execution with full observation
                logs.append({
                    "type": "tool_execution",
                    "iteration": iteration + 1,
                    "tool": action,
                    "query": action_input,
                    "citations_found": obs_citations,
                    "citations_count": len(obs_citations),
                    "observation": observation,
                    "observation_trunc": observation[:500] if len(observation) > 500 else observation,
                })
                
                if verbose:
                    print(f"\n[Tool: {action}]")
                    print(f"  Query: {action_input}")
                    print(f"  Citations found: {len(obs_citations)}")
                    if obs_citations:
                        print(f"  Citations: {obs_citations[:5]}")
                    print(f"  Observation (trunc): {observation[:300]}")
            else:
                error_msg = f"Unknown tool '{action}'. Available: search_laws, search_courts"
                observations.append(f"Tool {action_lower}: {error_msg}")
                logs.append({
                    "type": "tool_error",
                    "iteration": iteration + 1,
                    "tool": action,
                    "error": error_msg,
                })
        
        # Add all observations to conversation
        if observations:
            conversation += "\n" + "\n".join(observations) + "\n\n[INST] Continue your analysis. [/INST]\n\nThought:"
        
        # Check for final answer AFTER executing all actions
        if "Final Answer:" in response:
            final_text = response.split("Final Answer:")[-1].strip()
            citations = extract_citations_from_text(final_text)
            all_citations.extend(citations)
            
            logs.append({
                "type": "parse",
                "iteration": iteration + 1,
                "status": "final_answer_seen",
            })
            
            if verbose:
                print(f"\n[Iteration {iteration + 1}] Final Answer detected")
            break
        
        # If no actions found and no final answer, try to extract citations from response
        if not actions and "Final Answer:" not in response:
            citations = extract_citations_from_text(response)
            all_citations.extend(citations)
            logs.append({
                "type": "parse",
                "iteration": iteration + 1,
                "status": "no_actions_found",
                "citations_extracted": citations,
            })
            break
    
    # Deduplicate citations
    unique_citations = list(set(all_citations))
    
    logs.append({
        "type": "summary",
        "total_iterations": len(logs),
        "total_citations": len(unique_citations),
        "citations": unique_citations,
    })
    
    if verbose:
        print("\n" + "="*50)
        print("Found citations:")
        for c in unique_citations:
            print(f"  - {c}")
    
    return unique_citations, logs

In [15]:
# Optional prompt injection (type registry)
if CONFIG.get("prompt_injection_enabled", False):
    AGENT_SYSTEM_PROMPT += f"""

=== VERFÜGBARE RECHTSQUELLEN IM KORPUS ===

Gesetzestypen (häufigste, mit Dokumentanzahl):
{LAW_TYPES_FOR_PROMPT}

Gerichtsentscheid-Typen:
{COURT_TYPES_FOR_PROMPT}

HINWEIS: Du musst KEINE Gesetzesabkürzungen in deinen Suchanfragen verwenden.
Die Such-Tools erkennen automatisch den relevanten Rechtsbereich.
Beschreibe einfach das rechtliche Problem auf Deutsch."""

    print(f"Agent prompt updated with type registry")
else:
    print("Prompt injection disabled (Notebook-02 parity prompt).")

print(f"  Total prompt length: {len(AGENT_SYSTEM_PROMPT):,} chars")


Prompt injection disabled (Notebook-02 parity prompt).
  Total prompt length: 3,471 chars


In [16]:
# Test agent with a sample query
test_query = "What are the requirements for a valid contract under Swiss law?"
print(f"Query: {test_query}")
print("\nRunning agent...\n")

citations, logs = run_agent(test_query, verbose=True)

print("\n" + "="*50)
print("Found citations:")
for c in citations:
    print(f"  - {c}")

Query: What are the requirements for a valid contract under Swiss law?

Running agent...




[Iteration 1] LLM output (trunc):
 Ich suche nach Vertragsvoraussetzungen im Schweizer Recht.
Action: search_laws
Action Input: Vertrag Abschluss Voraussetzungen OR



[Iteration 1] Parsed 1 action(s):
  Action: search_laws, Input: Vertrag Abschluss Voraussetzungen OR

[Tool: search_laws]
  Query: Vertrag Abschluss Voraussetzungen OR
  Citations found: 40
  Citations: ['Art. 22 Abs. 1 OR', 'Art. 23 OR', 'Art. 20 Abs. 2 VEAGOG', 'Art. 22 Abs. 3 VEAGOG', 'Art. 18a Abs. 3 IVG']
  Observation (trunc): - Art. 22 Abs. 1 OR: 1 Durch Vertrag kann die Verpflichtung zum Abschluss eines künftigen Vertrages begründet werden.
- Art. 23 OR: Der Vertrag ist für denjenigen unverbindlich, der sich beim Abschluss in einem wesentlichen Irrtum befunden hat.
- Art. 20 Abs. 2 VEAGOG: 2 Der Leistungsauftrag wird mi

[Iteration 2] LLM output (trunc):
 Jetzt suche ich nach BGE-Entscheidungen zum Vertragsschluss.
Action: search_courts
Action Input: Vertragsabschluss Gültigkeit Voraussetzungen

Tool search_cour

## 8. Load Test Data

In [17]:
import pandas as pd

# Load queries from the configured query file
if not QUERY_FILE.exists():
    raise FileNotFoundError(f"Query file not found: {QUERY_FILE}")

test_df = pd.read_csv(QUERY_FILE)

print(f"Loaded {len(test_df)} queries from {QUERY_FILE}")
print(f"Columns: {list(test_df.columns)}")

if IS_VALIDATION_MODE and "gold_citations" in test_df.columns:
    print(f"Gold citations available for evaluation")

test_df.head()

Loaded 40 queries from /kaggle/input/competitions/llm-agentic-legal-information-retrieval/test.csv
Columns: ['query_id', 'query']


,query_id,query
0,test_001,Four U.S.-based software companies (NorthWave ...
1,test_002,On 9 August 2011 a 62‑year‑old cyclist (the cl...
2,test_003,"On 12 March 2012, Meridian Leasing Ltd and Ori..."
3,test_004,"A publicly listed manufacturing company, Orion..."
4,test_005,"A logistics company (R Ltd.), which owns a dis..."


## 9. Generate Predictions

In [ ]:
from tqdm import tqdm

# Generate predictions
predictions = []
all_logs = []  # Store logs for all queries

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Running agent"):
    query_id = row["query_id"]
    query_text = row["query"]
    
    # Run agent
    raw_citations, logs = run_agent(query_text, verbose=False)
    
    # Store logs with query_id
    all_logs.append({
        "query_id": query_id,
        "query": query_text,
        "logs": logs,
    })
    
    predictions.append({
        "query_id": query_id,
        "predicted_citations": ";".join(raw_citations),
    })

print(f"\nGenerated predictions for {len(predictions)} queries")



Running agent:  28%|██▊       | 11/40 [08:30<23:12, 48.02s/it]

In [ ]:
predictions_df = pd.DataFrame(predictions)
print(f"Predictions DataFrame: {predictions_df.shape}")
predictions_df.head()
print(f"Collected logs for {len(all_logs)} queries")

Predictions DataFrame: (10, 2)
Collected logs for 10 queries


## 10. Create Submission

In [ ]:
# Save submission (distinct filename from baseline)
submission_path = OUTPUT_PATH / "submission_hyde.csv"
predictions_df.to_csv(submission_path, index=False)

print(f"Submission saved to: {submission_path}")
print(f"Total predictions: {len(predictions_df)}")
print(f"HyDE enabled: {CONFIG['hyde_enabled']}")
print(f"HyDE cache hits: {len(_hyde_cache)}")

# Show sample
print("\nSample submission:")
print(predictions_df.head())

Submission saved to: C:\Users\6764325\OneDrive - MyFedEx\Desktop\Full Pipeline - RAG Competition\Omnilex-Agentic-Retrieval-Competition\output\submission.csv
Total predictions: 10

Sample submission:
  query_id                                predicted_citations
0  val_001  Art. 56 Abs. 2 MStP;Art. 72 Abs. 3 V-ASG;Art. ...
1  val_002  BGE 134 V 170 E. 1;BGE 134 IV 315 E. 4.2.2;BGE...
2  val_003  BGE 145 IV 424 E. 4.1;BGE 144 IV 28 E. 1.3.2;B...
3  val_004  BGE 137 III 539 E. 5.2;BGE 142 III 502 E. 2.7;...
4  val_005  Art. 63 Abs. 4 AsylG;Art. 7 MSG;Art. 296 Abs. ...


In [20]:
from collections.abc import Sequence


def citation_f1(
    predicted: Sequence[str],
    gold: Sequence[str],
) -> dict[str, float]:
    """Compute F1 score for citation overlap on a single query.

    Args:
        predicted: List of predicted canonical citation IDs
        gold: List of ground truth canonical citation IDs

    Returns:
        Dictionary with precision, recall, and F1
    """
    pred_set = set(predicted)
    gold_set = set(gold)

    # Edge case: both empty
    if len(pred_set) == 0 and len(gold_set) == 0:
        return {"precision": 1.0, "recall": 1.0, "f1": 1.0}

    # Edge case: prediction empty but gold not
    if len(pred_set) == 0:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}

    # Edge case: gold empty but prediction not
    if len(gold_set) == 0:
        return {"precision": 0.0, "recall": 1.0, "f1": 0.0}

    true_positives = len(pred_set & gold_set)
    precision = true_positives / len(pred_set)
    recall = true_positives / len(gold_set)

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return {"precision": precision, "recall": recall, "f1": f1}


def macro_f1(
    predictions: Sequence[Sequence[str]],
    gold: Sequence[Sequence[str]],
) -> dict[str, float]:
    """Compute Macro F1: average F1 across all queries.

    This is the PRIMARY competition metric.

    Args:
        predictions: List of predicted citation lists (one per query)
        gold: List of gold citation lists (one per query)

    Returns:
        Dictionary with macro precision, recall, and F1
    """
    if len(predictions) != len(gold):
        raise ValueError(f"Length mismatch: {len(predictions)} predictions vs {len(gold)} gold")

    if len(predictions) == 0:
        return {"macro_precision": 0.0, "macro_recall": 0.0, "macro_f1": 0.0}

    precision_scores = []
    recall_scores = []
    f1_scores = []

    for pred, g in zip(predictions, gold):
        scores = citation_f1(pred, g)
        precision_scores.append(scores["precision"])
        recall_scores.append(scores["recall"])
        f1_scores.append(scores["f1"])

    n = len(f1_scores)
    return {
        "macro_precision": sum(precision_scores) / n,
        "macro_recall": sum(recall_scores) / n,
        "macro_f1": sum(f1_scores) / n,
    }


def micro_f1(
    predictions: Sequence[Sequence[str]],
    gold: Sequence[Sequence[str]],
) -> dict[str, float]:
    """Compute Micro F1: aggregate TP/FP/FN across all queries.

    Args:
        predictions: List of predicted citation lists (one per query)
        gold: List of gold citation lists (one per query)

    Returns:
        Dictionary with micro precision, recall, and F1
    """
    if len(predictions) != len(gold):
        raise ValueError(f"Length mismatch: {len(predictions)} predictions vs {len(gold)} gold")

    total_tp = 0
    total_fp = 0
    total_fn = 0

    for pred, g in zip(predictions, gold):
        pred_set = set(pred)
        gold_set = set(g)

        tp = len(pred_set & gold_set)
        fp = len(pred_set - gold_set)
        fn = len(gold_set - pred_set)

        total_tp += tp
        total_fp += fp
        total_fn += fn

    if total_tp + total_fp == 0:
        precision = 0.0
    else:
        precision = total_tp / (total_tp + total_fp)

    if total_tp + total_fn == 0:
        recall = 0.0
    else:
        recall = total_tp / (total_tp + total_fn)

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return {
        "micro_precision": precision,
        "micro_recall": recall,
        "micro_f1": f1,
    }


def evaluate_submission(
    submission_df: pd.DataFrame,
    gold_df: pd.DataFrame,
    metrics: list[str] | None = None,
) -> dict[str, float]:
    """Evaluate a submission DataFrame against gold DataFrame.

    Args:
        submission_df: DataFrame with query_id and predicted_citations
        gold_df: DataFrame with query_id and gold_citations
        metrics: List of metrics to compute (default: all)

    Returns:
        Dictionary with requested metric scores
    """
    citation_separator = ";"
    
    def parse_citations(citation_string: str) -> list[str]:
        """Parse citation string into list (citations are already normalized)."""
        if not citation_string or citation_string.strip() == "":
            return []
        return [c.strip() for c in citation_string.split(citation_separator) if c.strip()]

    # Merge DataFrames
    merged = pd.merge(
        submission_df,
        gold_df,
        on="query_id",
        how="inner",
    )

    # Parse citations
    predictions = [
        parse_citations(row.get("predicted_citations", "")) for _, row in merged.iterrows()
    ]
    gold = [parse_citations(row.get("gold_citations", "")) for _, row in merged.iterrows()]

    # Compute all scores
    all_scores = {}

    macro_scores = macro_f1(predictions, gold)
    micro_scores = micro_f1(predictions, gold)

    all_scores.update(macro_scores)
    all_scores.update(micro_scores)

    # Log per-sample TP/FP/FN for each query
    print("\n" + "="*50)
    print("PER-SAMPLE EVALUATION RESULTS")
    print("="*50)
    for idx, (_, row) in enumerate(merged.iterrows()):
        query_id = row["query_id"]
        pred_set = set(predictions[idx])
        gold_set = set(gold[idx])
        
        true_positives = list(pred_set & gold_set)
        false_positives = list(pred_set - gold_set)
        false_negatives = list(gold_set - pred_set)
        
        print(f"\nQuery ID: {query_id}")
        print(f"  True Positives ({len(true_positives)}): {true_positives}")
        print(f"  False Positives ({len(false_positives)}): {false_positives}")
        print(f"  False Negatives ({len(false_negatives)}): {false_negatives}")
    
    print("\n" + "="*50)

    # Filter to requested metrics
    if metrics:
        metric_mapping = {
            "f1": "macro_f1",
            "precision": "macro_precision",
            "recall": "macro_recall",
            "macro_f1": "macro_f1",
            "micro_f1": "micro_f1",
        }
        filtered = {}
        for m in metrics:
            key = metric_mapping.get(m, m)
            if key in all_scores:
                filtered[m] = all_scores[key]
        return filtered

    return all_scores

## 11. Local Evaluation (Optional)

In [21]:
# Evaluate if in validation mode with gold labels
if IS_VALIDATION_MODE and "gold_citations" in test_df.columns:
    # Join predictions with gold citations from the same file
    eval_df = predictions_df.merge(
        test_df[["query_id", "gold_citations"]],
        on="query_id",
        how="inner"
    )
    
    if len(eval_df) > 0:
        scores = evaluate_submission(
            eval_df[["query_id", "predicted_citations"]],
            eval_df[["query_id", "gold_citations"]],
        )
        
        print("\n" + "="*50)
        print("EVALUATION RESULTS")
        print("="*50)
        print(f"Queries evaluated: {len(eval_df)}")
        print(f"\nMacro F1 (PRIMARY): {scores['macro_f1']:.4f}")
        print(f"Macro Precision:    {scores['macro_precision']:.4f}")
        print(f"Macro Recall:       {scores['macro_recall']:.4f}")
        print(f"\nMicro F1:           {scores['micro_f1']:.4f}")
        print(f"Micro Precision:    {scores['micro_precision']:.4f}")
        print(f"Micro Recall:       {scores['micro_recall']:.4f}")
    else:
        print("No overlapping queries for evaluation.")
else:
    print("Skipping evaluation (not in validation mode or no gold labels available)")


PER-SAMPLE EVALUATION RESULTS

Query ID: val_001
  True Positives (2): ['Art. 39 Abs. 1 StBOG', 'BGE 137 IV 122 E. 4.2']
  False Positives (118): ['Art. 36 Abs. 2 BÜPF', 'Art. 16 Abs. 5 VKV-FINMA', 'Art. 6 Abs. 4 BStKR', 'BGE 121 I 208 E. 4b', 'BGE 137 IV 237 E. 2.2', 'BGE 133 IV 150 E. 5.1', 'Art. 440 Abs. 1 StPO', 'Art. 76 Abs. 2 GBV', 'Art. 45 Abs. 1 MStV', 'Art. 226 Abs. 5 StPO', 'Art. 406 Abs. 2 StPO', 'BGE 146 IV 279 E. 2.2', 'BGE 137 IV 177 E. 2.1', 'BGE 145 IV 424 E. 4.5.1', 'Art. 55 MStV', 'Art. 24 Abs. 2 StReG', 'BGE 128 I 149 E. 2.1', 'Art. 354 Abs. 2 StPO', 'Art. 37 Abs. 2 BÜPF', 'Art. 84c Abs. 3 MStP', 'Art. 21 Abs. 4 ZISG', 'Art. 15d Abs. 1 813.121', 'Art. 1 Abs. 2 313.32', 'BGE 137 IV 237 E. 2.4', 'Art. 58 Abs. 3 VStrR', 'BGE 148 IV 419 E. 1.7.2', 'BGE 146 IV 279 E. 3.2', 'Art. 65 Abs. 2 RTVG', 'Art. 27 Abs. 5 JStPO', 'Art. 65 Abs. 3 MStP', 'Art. 80 Abs. 1 VStrR', 'Art. 8 Abs. 2 BStKR', 'Art. 73 Abs. 3 VStrR', 'Art. 327 Abs. 1 StPO', 'Art. 82 KAG', 'Art. 229 Abs. 3 StPO

## Summary

This HyDE-enhanced retrieval notebook adds **Hypothetical Document Embedding** on top of the agentic baseline:

1. **Few-shot bank from train.csv + synthetic fill**: Gold citations resolved to actual corpus text, with synthetic queries generated for types lacking training examples. Banks are cached to pickle for fast reload.
2. **Keyword-based type detection**: Each German query is scored against the bank's `query_en` fields by word overlap to detect relevant legal types — no LLM routing needed.
3. **HyDE document generation**: Before each BM25 search, the LLM generates a hypothetical German legal passage using cross-lingual few-shot examples (German query → German query → German text).
4. **Single hierarchical search**: BM25 search with the HyDE document, soft-boosting documents matching the detected type(s).
5. **CCH-style labels**: Results include `[TYPE]` headers for LLM context awareness across agent iterations.

## Architecture: Per-Tool-Call Pipeline

```
German query → keyword matching (select_few_shot_examples)
                  → type_hints + matched examples
                → build_hyde_prompt (German few-shot examples)
                  → Mistral generates German hypothetical document
                → hierarchical_bm25_search (single search + type boost)
                  → CCH-labeled results → Observation
```

Each tool call is independent — type detection is re-derived from the raw query, not carried forward from previous observations.

## Ablation Testing

Set `CONFIG["hyde_enabled"] = False` to run in keyword-only mode for comparison.

## Potential Further Improvements

- **Multiple HyDE documents**: Generate 2-3 hypothetical docs per query for higher recall
- **Embedding-based search**: Replace/augment BM25 with vector similarity using HyDE embeddings
- **Iterative HyDE**: Generate new hypothetical documents in later agent iterations based on already-found results
- **Observation feedback**: Pass type information from previous iterations' observations into subsequent tool calls

In [ ]:
# Load test set
TEST_QUERY_FILE = DATA_PATH / "test.csv"

if TEST_QUERY_FILE.exists():
    print(f"Loading test set from {TEST_QUERY_FILE}")
    test_set_df = pd.read_csv(TEST_QUERY_FILE)
    print(f"Loaded {len(test_set_df)} test queries")
    print(f"Columns: {list(test_set_df.columns)}")
    
    # Generate predictions for test set
    test_predictions = []
    test_all_logs = []  # Store logs for all test queries
    
    print("\n" + "="*50)
    print("RUNNING AGENT ON TEST SET")
    print("="*50)
    
    for _, row in tqdm(test_set_df.iterrows(), total=len(test_set_df), desc="Running agent on test set"):
        query_id = row["query_id"]
        query_text = row["query"]
        
        # Run agent
        raw_citations, logs = run_agent(query_text, verbose=False)
        
        # Store logs with query_id
        test_all_logs.append({
            "query_id": query_id,
            "query": query_text,
            "logs": logs,
        })
        
        test_predictions.append({
            "query_id": query_id,
            "predicted_citations": ";".join(raw_citations),
        })
    
    print(f"\nGenerated predictions for {len(test_predictions)} test queries")
    print(f"Collected logs for {len(test_all_logs)} test queries")
    
    # Create DataFrame and save test submission
    test_predictions_df = pd.DataFrame(test_predictions)
    test_submission_path = OUTPUT_PATH / "test_submission.csv"
    test_predictions_df.to_csv(test_submission_path, index=False)
    
    print(f"\nTest submission saved to: {test_submission_path}")
    print(f"Total test predictions: {len(test_predictions_df)}")
    print("\nSample test submission:")
    print(test_predictions_df.head())
else:
    print(f"Test set file not found: {TEST_QUERY_FILE}")
    print("Skipping test set processing.")

Loading test set from C:\Users\6764325\OneDrive - MyFedEx\Desktop\Full Pipeline - RAG Competition\Omnilex-Agentic-Retrieval-Competition\data\test.csv
Loaded 40 test queries
Columns: ['query_id', 'query']

RUNNING AGENT ON TEST SET


Running agent on test set:  20%|██        | 8/40 [7:04:19<47:06:56, 5300.52s/it]